# Building Knowledge Graphs with LLMs

**Course module:** Module 8
**Audience:** Beginners who already understand graph basics, Neo4j fundamentals,
Generative AI / LLMs, and simple Cypher.

**Course description**

In this hands-on course, you will learn how to create and query knowledge graphs using
Large Language Models (LLMs). You will use the **Neo4j LLM Knowledge Graph Builder** to
build knowledge graphs from unstructured data, then customize the data model and query
the graph with **Cypher**

**What you will learn**

1. What a knowledge graph is and how it supports GenAI applications (GraphRAG, grounded answers).
2. How an LLM can extract **entities** and **relationships** from unstructured text.
3. How to use the **LLM Knowledge Graph Builder** (web app) for rapid prototyping.
4. How to build and load a graph **programmatically** from Python (same ideas as the web app).
5. How to **guide extraction with a schema** and **query** results with Cypher.

> **Language:** All instructional text in this course is in **English**.

> **Setup:** Complete **`NEO4J_SETUP.md`** (Neo4j/database) and **`LLM_MODEL_SETUP.md`** (LLM via `ollama_model_runner.py` or OpenAI) in this folder before running code cells.

## Course outline

| Part | Topic |
|------|--------|
| **01** | Knowledge graphs |
| **02** | LLM Graph Builder |
| **03** | Exploring your knowledge graph |
| 0 | Environment & Neo4j connection |

### 01. Knowledge graphs

| Section | Topic |
|---------|--------|
| 1.1 | What is a Knowledge Graph |
| 1.2 | Benefits and challenges |
| 1.3 | Explore a Knowledge Graph |
| 1.4 | Knowledge Graph Use Cases |

### 02. LLM Graph Builder

| Section | Topic |
|---------|--------|
| 2.1 | How to Construct a Knowledge Graph with an LLM |
| 2.2 | Neo4j LLM Graph Builder |
| 2.3 | Customize the schema |
| 2.4 | Complete the Graph |

### 03. Exploring your knowledge graph

| Section | Topic |
|---------|--------|
| 3.1 | Querying using Cypher |
| 3.2 | Explore the Knowledge Graph |
| 3.3 | Create your Knowledge Graph |

## Step 0 — Environment and Neo4j connection
### Before you run any code
Follow **`NEO4J_SETUP.md`** and **`LLM_MODEL_SETUP.md`** end to end. You need:
- A running Neo4j 5.15+ database (Aura, Desktop, or Docker)
- Connection URI, username, password
- **Ollama** + `ollama_model_runner.py` (recommended), or **OpenAI**
### What this step does
1. Install Python packages.
2. Load Neo4j and LLM settings from `Module_8/.env`.
3. Verify Neo4j connectivity.
4. Configure the LLM for `LLMGraphTransformer` (Steps **0d.1–0d.4**):
   - **`LLM_BACKEND=ollama`** → subprocess calls `ollama_model_runner.py`
   - **`LLM_BACKEND=openai`** → `ChatOpenAI`


In [1]:
# Step 0a — Install Python dependencies (run once per environment)
%pip install -q neo4j python-dotenv requests \
    langchain langchain-community langchain-experimental langchain-neo4j \
    langchain-openai sentence-transformers



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Step 0b.1 — Resolve the `Module_8` folder
Jupyter may start in the repo root or inside `Module_8`. This cell finds the folder that contains `.env` and `data/`.


In [2]:
# Step 0b.1 — Resolve Module_8 directory
import os
from pathlib import Path
from dotenv import load_dotenv
MODULE_DIR = Path(".").resolve()
if MODULE_DIR.name != "Module_8":
    _candidate = MODULE_DIR / "Module_8"
    if _candidate.is_dir():
        MODULE_DIR = _candidate.resolve()
load_dotenv(MODULE_DIR / ".env")
print("Module directory:", MODULE_DIR)


Module directory: /home/ethan/newgen/KMOU_Course/Module_8


### Step 0b.2 — Neo4j connection settings
Values come from `Module_8/.env` (see **`NEO4J_SETUP.md`**).


In [3]:
# Step 0b.2 — Neo4j settings
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
print("Neo4j URI:", NEO4J_URI)
print("Neo4j database:", NEO4J_DATABASE)


Neo4j URI: neo4j://localhost:7687
Neo4j database: neo4j


### Step 0b.3 — LLM backend and lab dataset
- **`LLM_BACKEND=ollama`** (default) uses `ollama_model_runner.py` in later cells.
- **`LLM_BACKEND=openai`** uses `ChatOpenAI` and requires `OPENAI_API_KEY`.
Default corpus: **`data/dbpedia_course_corpus.txt`** (see `data/DATASETS.md`).

- **`LAB_MAX_ARTICLES`** (default `5`): how many corpus articles to send to the LLM in Section 2.2b (avoids 100× prompts / huge single prompts).
- **`LAB_SCHEMA_ARTICLES`** (default `8`): articles used in Section 2.3; maritime entries work best for the Port/Organization schema.



In [4]:
# Step 0b.3 — LLM + dataset path
LLM_BACKEND = os.getenv("LLM_BACKEND", "ollama").lower()
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
OLLAMA_TEMPERATURE = float(os.getenv("OLLAMA_TEMPERATURE", "0"))
OLLAMA_MAX_TOKENS = int(os.getenv("OLLAMA_MAX_TOKENS", "2048"))
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
LAB_MAX_ARTICLES = int(os.getenv("LAB_MAX_ARTICLES", "5"))
LAB_SCHEMA_ARTICLES = int(os.getenv("LAB_SCHEMA_ARTICLES", "8"))
SAMPLE_TEXT_PATH = MODULE_DIR / "data" / "dbpedia_course_corpus.txt"
# SAMPLE_TEXT_PATH = MODULE_DIR / "data" / "dbpedia_maritime_corpus.txt"
print("LLM backend:", LLM_BACKEND)
print("LAB_MAX_ARTICLES (open schema):", LAB_MAX_ARTICLES)
print("LAB_SCHEMA_ARTICLES (schema lab):", LAB_SCHEMA_ARTICLES)
print("Dataset:", SAMPLE_TEXT_PATH.name, "| exists:", SAMPLE_TEXT_PATH.is_file())
if LLM_BACKEND == "ollama":
    print("Ollama host:", OLLAMA_HOST)
    print("Ollama model:", OLLAMA_MODEL)



LLM backend: ollama
LAB_MAX_ARTICLES (open schema): 5
LAB_SCHEMA_ARTICLES (schema lab): 8
Dataset: dbpedia_course_corpus.txt | exists: True
Ollama host: http://localhost:11434
Ollama model: llama3.1:8b


In [5]:
# Step 0c — Verify Neo4j connectivity
from neo4j import GraphDatabase

if not NEO4J_PASSWORD:
    raise ValueError(
        "NEO4J_PASSWORD is empty. Set it in your environment or Module_8/.env "
        "(see NEO4J_SETUP.md and LLM_MODEL_SETUP.md)."
    )

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with driver.session(database=NEO4J_DATABASE) as session:
    record = session.run('RETURN "Neo4j connection OK" AS message').single()
    print(record["message"])

driver.close()
print("Connectivity check passed.")

Neo4j connection OK
Connectivity check passed.


### Step 0d.1 — Locate `ollama_model_runner.py`
The course reuses the same subprocess runner as Module 4/5. We search `Module_8/`, then `Module_4/`, then the current directory.


In [6]:
# Step 0d.1 — Resolve runner script path
import json
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import Any, List, Optional
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.llms import LLM
def _resolve_ollama_runner_path() -> Path:
    for candidate in (
        MODULE_DIR / "ollama_model_runner.py",
        MODULE_DIR.parent / "Module_4" / "ollama_model_runner.py",
        Path("ollama_model_runner.py"),
    ):
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "ollama_model_runner.py not found. Expected in Module_8/ (see LLM_MODEL_SETUP.md)."
    )
OLLAMA_RUNNER = _resolve_ollama_runner_path()
print("OLLAMA_RUNNER:", OLLAMA_RUNNER)


/home/ethan/anaconda3/envs/kmouc/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OLLAMA_RUNNER: /home/ethan/newgen/KMOU_Course/Module_8/ollama_model_runner.py


### Step 0d.2 — `call_ollama_runner()` helper
1. Write the prompt to a temporary file.
2. Invoke `python ollama_model_runner.py` with host, model, temperature, and max tokens.
3. Parse JSON stdout and return the first successful `response` string.
This keeps Ollama configuration in one script instead of duplicating HTTP calls in the notebook.


In [7]:
# Step 0d.2 — Subprocess wrapper around ollama_model_runner.py
def call_ollama_runner(prompt: str, *, model: str | None = None) -> str:
    """Call ollama_model_runner.py (same pattern as Module 4 / Module 5)."""
    model_arg = model or OLLAMA_MODEL
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".txt", delete=False, encoding="utf-8"
    ) as pf:
        path = pf.name
        pf.write(prompt)
    try:
        cmd = [
            sys.executable,
            str(OLLAMA_RUNNER),
            "--host",
            OLLAMA_HOST,
            "--models",
            model_arg,
            "--prompt-file",
            path,
            "--temperature",
            str(OLLAMA_TEMPERATURE),
            "--max-tokens",
            str(OLLAMA_MAX_TOKENS),
        ]
        run = subprocess.run(cmd, capture_output=True, text=True)
        if run.returncode != 0:
            raise RuntimeError(
                f"ollama_model_runner.py exit {run.returncode}\nSTDERR:\n{run.stderr[:4000]}"
            )
        payload = json.loads(run.stdout)
        outputs = payload.get("outputs") or []
        if not outputs:
            raise RuntimeError("Runner JSON contained no outputs")
        first = outputs[0]
        if first.get("status") != "ok":
            raise RuntimeError(f"Ollama runner error: {first}")
        return (first.get("response") or "").strip()
    finally:
        Path(path).unlink(missing_ok=True)


### Step 0d.3 — LangChain `OllamaRunnerLLM` class
`LLMGraphTransformer` expects a LangChain **`LLM`** (or chat model). This thin wrapper forwards `_call()` to `call_ollama_runner()`.


In [8]:
# Step 0d.3 — LangChain LLM adapter
class OllamaRunnerLLM(LLM):
    """LangChain LLM that delegates to ollama_model_runner.py."""
    model: str = OLLAMA_MODEL
    temperature: float = OLLAMA_TEMPERATURE
    max_tokens: int = OLLAMA_MAX_TOKENS
    @property
    def _llm_type(self) -> str:
        return "ollama_runner"
    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        return call_ollama_runner(prompt, model=self.model)


### Step 0d.4 — Create the `llm` object used later
Run all Step 0d cells before extraction. For Ollama, ensure `ollama serve` is running and the model is pulled (`ollama pull ...`).


In [9]:
# Step 0d.4 — Select backend and instantiate llm
if LLM_BACKEND == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("OPENAI_API_KEY is required when LLM_BACKEND=openai")
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    print(f"Using OpenAI model: {OPENAI_MODEL}")
elif LLM_BACKEND == "ollama":
    llm = OllamaRunnerLLM()
    print(f"Using Ollama via runner: {OLLAMA_MODEL} @ {OLLAMA_HOST}")
    print("Ensure Ollama is running: ollama serve")
else:
    raise ValueError("Set LLM_BACKEND to 'ollama' or 'openai'")


Using Ollama via runner: llama3.1:8b @ http://localhost:11434
Ensure Ollama is running: ollama serve


### Step 0e — Smoke test `ollama_model_runner.py`

Run this once before graph extraction. If it fails, fix Ollama setup in **`LLM_MODEL_SETUP.md`**.


In [10]:
# Step 0e — Smoke test (ollama backend only)
if LLM_BACKEND == "ollama":
    _smoke = call_ollama_runner("Reply with exactly: Ollama OK", model=OLLAMA_MODEL)
    print("Ollama runner smoke test:", _smoke[:200])
    if "ok" not in _smoke.lower():
        print("Warning: unexpected smoke reply — check model and OLLAMA_HOST")
    print("LLM ready for LLMGraphTransformer (backend=ollama)")
else:
    print("Skipped — OpenAI backend selected")


Ollama runner smoke test: Ollama OK
LLM ready for LLMGraphTransformer (backend=ollama)


---

# 01. Knowledge graphs

## 1.1 What is a Knowledge Graph

A **knowledge graph** is a graph database representation of **facts** about your domain:

- **Nodes** = entities (e.g. `Port`, `Organization`, `Canal`, `Country`)
- **Relationships** = typed connections (e.g. `LOCATED_IN`, `OPERATED_BY`, `CONNECTS`)
- **Properties** = attributes on nodes or relationships (e.g. `name`, `country`, `throughput`)

### Unstructured vs structured

| Unstructured | Structured (in Neo4j) |
|--------------|-------------------------|
| Paragraphs in PDFs, emails, reports | Labeled nodes and relationships |
| Hard to query with SQL alone | Easy to traverse with Cypher |
| Ambiguous references ("the port") | Explicit entities (`Port_of_Rotterdam`) |

### How this course uses knowledge graphs

You will start from **unstructured text** (DBpedia-style abstracts), use an **LLM** to propose entities and relationships, store them in **Neo4j**, then **query and explore** the resulting graph with Cypher.

## 1.2 Benefits and challenges

### Benefits

| Benefit | Why it matters |
|---------|----------------|
| **Connected context** | Relationships encode meaning beyond isolated text chunks |
| **Explainability** | Answers can be traced along graph paths |
| **Reusable data model** | Stable labels and relationship types for apps and APIs |
| **Better GenAI (GraphRAG)** | Combine vector search on chunks with graph traversal |

### Challenges

| Challenge | Practical impact |
|-----------|------------------|
| **Schema design** | Inconsistent labels reduce query quality |
| **Extraction quality** | LLMs may hallucinate entities or relations |
| **Data cleaning** | Duplicate entities and ambiguous names need review |
| **Operational cost** | LLM extraction and embedding generation take time and compute |

In this module you will reduce these risks using **schema-guided extraction** and **Cypher validation queries**.

## 1.3 Explore a Knowledge Graph

Before building a graph from unstructured text, explore a **small reference graph** in Neo4j.

### Learning goals

- Run basic Cypher `MATCH` patterns
- Read node labels, relationship types, and properties
- Visualize a subgraph in Neo4j Browser

> Run the next cell after Step 0 (Neo4j connection is working).

### 1.3a — Seed reference nodes (safe to re-run)
We create a **tiny** maritime-themed graph: ports, an organization, a canal, and countries. A `CourseKG` marker node tags this lab data for optional cleanup later.


In [11]:
# 1.3a — Seed reference graph (write)
from neo4j import GraphDatabase
seed_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
seed_cypher = [
    '''
    MERGE (demo:CourseKG {module: 'Module_8', lab: 'reference_explore'})
    ''',
    '''
    MERGE (p1:Port {id: 'Port_of_Rotterdam'})
    SET p1.country = 'Netherlands'
    MERGE (p2:Port {id: 'Port_of_Busan'})
    SET p2.country = 'South Korea'
    MERGE (o:Organization {id: 'Maersk'})
    SET o.country = 'Denmark'
    MERGE (c:Canal {id: 'Panama_Canal'})
    SET c.country = 'Panama'
    MERGE (p1)-[:LOCATED_IN]->(country1:Country {id: 'Netherlands'})
    MERGE (p2)-[:LOCATED_IN]->(country2:Country {id: 'South Korea'})
    MERGE (o)-[:OPERATES_IN]->(p1)
    MERGE (o)-[:USES_ROUTE]->(c)
    ''',
]
with seed_driver.session(database=NEO4J_DATABASE) as session:
    for q in seed_cypher:
        session.run(q)
print("Reference graph seeded.")


Reference graph seeded.


### 1.3b — Inspect seeded nodes and relationships
Confirm data in the notebook, then open **Neo4j Browser** and visualize paths, e.g.:
```cypher
MATCH p=(:Port)-[*1..2]-() RETURN p LIMIT 25
```


In [12]:
# 1.3b — Query reference graph
with seed_driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run(
        '''
        MATCH (n)
        WHERE n:Port OR n:Organization OR n:Canal OR n:Country
        RETURN labels(n) AS labels, n.id AS id
        ORDER BY id
        LIMIT 20
        '''
    ).data()
    rels = session.run(
        '''
        MATCH (a)-[r]->(b)
        WHERE type(r) IN ['LOCATED_IN', 'OPERATES_IN', 'USES_ROUTE']
        RETURN a.id AS from, type(r) AS rel, b.id AS to
        LIMIT 20
        '''
    ).data()
seed_driver.close()
print("Nodes:")
for row in rows:
    print(" ", row)
print("Relationships:")
for row in rels:
    print(" ", row)


Nodes:
  {'labels': ['Organization'], 'id': 'A.P. Moller - Maersk'}
  {'labels': ['Organization'], 'id': 'Busan Port Authority'}
  {'labels': ['Organization'], 'id': 'Congress'}
  {'labels': ['Country'], 'id': 'Egypt'}
  {'labels': ['Organization'], 'id': 'International Maritime Organization'}
  {'labels': ['Organization'], 'id': 'Maersk'}
  {'labels': ['Organization', 'LlamaIndexLab'], 'id': 'Maersk'}
  {'labels': ['Organization', 'LangChainLab'], 'id': 'Maersk'}
  {'labels': ['Organization'], 'id': 'Maersk Line'}
  {'labels': ['Organization'], 'id': 'Mediterranean Shipping Company'}
  {'labels': ['Country', 'LlamaIndexLab'], 'id': 'Netherlands'}
  {'labels': ['Country', 'LangChainLab'], 'id': 'Netherlands'}
  {'labels': ['Country'], 'id': 'Netherlands'}
  {'labels': ['Organization'], 'id': 'PSA International'}
  {'labels': ['Country'], 'id': 'Panama'}
  {'labels': ['Canal'], 'id': 'Panama Canal'}
  {'labels': ['Organization'], 'id': 'Panama Canal Authority'}
  {'labels': ['Canal', 'L

## 1.4 Knowledge Graph Use Cases

Knowledge graphs are used across industries when **relationships matter**:

| Use case | Example question |
|----------|------------------|
| **Search & discovery** | Which organizations operate ports in Europe? |
| **Recommendation** | Which products are frequently bought together? |
| **Risk & compliance** | Which suppliers are connected to a flagged entity? |
| **Operations** | Which assets depend on a disrupted canal route? |
| **GenAI / GraphRAG** | Answer questions with graph context, not only similar text |

### GenAI-specific use case (this course)

- Build a **lexical graph** (documents/chunks + embeddings)
- Build an **entity graph** (extracted nodes and relationships)
- Link chunks to entities for grounded retrieval

The Neo4j **LLM Knowledge Graph Builder** automates this pipeline; you will also build the same flow in Python.

---

# 02. LLM Graph Builder

## 2.1 How to Construct a Knowledge Graph with an LLM

A typical LLM-based construction pipeline:

1. **Ingest** unstructured sources (text files, PDFs, web pages)
2. **Chunk** text for processing and embeddings
3. **Extract** entities and relationships with an LLM
4. **Normalize** labels to your schema (optional but recommended)
5. **Load** nodes and relationships into Neo4j
6. **Validate** with Cypher and visual exploration

```mermaid
flowchart LR
  A[Unstructured text] --> B[Chunking]
  B --> C[LLM extraction]
  C --> D[Schema mapping]
  D --> E[(Neo4j graph)]
  E --> F[Cypher exploration]
```

### Dataset for this lab

| File | Description |
|------|-------------|
| **`data/dbpedia_course_corpus.txt`** | **Default** — 100 English DBpedia abstracts (~100 KB) |
| `data/dbpedia_maritime_corpus.txt` | 37 maritime/logistics-focused articles |
| `data/DATASETS.md` | Sources, licenses, rebuild & download scripts |

Source: [DBpedia-Summarizer en_100_summaries.csv](https://github.com/dice-group/DBpedia-Summarizer/blob/master/data_eval/en_100_summaries.csv) (CC BY-SA).

Rebuild corpora locally:

```bash
python Module_8/scripts/build_dbpedia_corpus.py
```

Optional larger maritime download (requires internet):

```bash
python Module_8/scripts/download_dbpedia_maritime_sparql.py --limit 150
```

Look for named entities, organizations, locations, and implied relationships ("headquartered in", "operated by", "managed by").


### Step 2a — Load the lab corpus

We read the plain-text DBpedia corpus from `SAMPLE_TEXT_PATH` (set in Step 0b.3). The preview helps you confirm encoding and approximate size before LLM extraction.



In [13]:
# Step 2a — Read sample unstructured text
raw_text = SAMPLE_TEXT_PATH.read_text(encoding="utf-8")
print(f"Characters: {len(raw_text)}")
print("--- Preview (first 500 chars) ---")
print(raw_text[:500])

Characters: 99962
--- Preview (first 500 chars) ---
DBpedia Course Corpus (100 articles) — Module 8 Knowledge Graph Lab

License note: Abstracts derived from DBpedia / Wikipedia content (CC BY-SA).
Source package: dice-group/DBpedia-Summarizer (en_100_summaries.csv) + curated maritime entries.

[1] 90_BC
Year 90 BC was a year of the pre-Julian Roman calendar. At the time it was known as the Year of the Consulship of Caesar and Lupus (or, less frequently, year 664 Ab urbe condita). The denomination 90 BC for this year has been used since the early


### Step 2a-helper — Split corpus into articles

Corpus files use markers like `[1] Port_of_Rotterdam`. **Do not** pass the full 100-article file as one `Document` with Ollama — the model output is truncated and you often get **0 nodes**. We build one `Document` per article and cap how many are processed in the lab.



In [14]:
# Step 2a-helper — Parse numbered articles from corpus text
import re


def parse_corpus_articles(text: str) -> list[dict]:
    """Split Module_8 corpus files on [N] Title lines."""
    pattern = re.compile(r"^\[(\d+)\]\s+(.+)$", re.MULTILINE)
    matches = list(pattern.finditer(text))
    if not matches:
        raise ValueError(
            "No [N] Title markers found. Expected dbpedia_*_corpus.txt format."
        )
    articles = []
    for i, m in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[m.end() : end].strip()
        articles.append(
            {
                "index": int(m.group(1)),
                "title": m.group(2).strip(),
                "text": f"{m.group(0).strip()}\n\n{body}",
            }
        )
    return articles


def articles_to_documents(
    articles: list[dict],
    *,
    source_name: str,
    max_articles: int,
    course: str = "Module_8",
) -> list:
    from langchain_core.documents import Document

    limited = articles[:max_articles]
    return [
        Document(
            page_content=a["text"],
            metadata={
                "source": source_name,
                "course": course,
                "article_index": a["index"],
                "title": a["title"],
            },
        )
        for a in limited
    ]


_all_articles = parse_corpus_articles(raw_text)
print(f"Parsed {len(_all_articles)} articles from {SAMPLE_TEXT_PATH.name}")
print("First titles:", [a["title"] for a in _all_articles[:5]])



Parsed 100 articles from dbpedia_course_corpus.txt
First titles: ['90_BC', 'Heinrich_Hertz', 'Olof_Palme', 'Seventh_Amendment_to_the_United_States_Constitution', 'Red-eye_effect']


---

## 2.2 Neo4j LLM Graph Builder (web application)

This step is **manual** (browser). It teaches the same pipeline you will code in Section 2.2b.

### 3.1 Open the application

1. Overview: [Neo4j Labs — LLM Graph Builder](https://neo4j.com/labs/genai-ecosystem/llm-graph-builder/)
2. Hosted app: [https://llm-graph-builder.neo4jlabs.com/](https://llm-graph-builder.neo4jlabs.com/)

### 3.2 Connect Neo4j

Enter URI, username, and password from **`NEO4J_SETUP.md`**, or drag your Aura credentials file. For LLM provider/model settings, follow **`LLM_MODEL_SETUP.md`**.

### 3.3 Ingest unstructured data

Upload `data/dbpedia_course_corpus.txt` (or `dbpedia_maritime_corpus.txt`) (downloaded from DBpedia pages) or paste the text. Wait for status **New**.

### 3.4 Configure LLM and schema (optional)

Select your LLM provider. Optionally define allowed node labels and relationship types.

### 3.5 Generate the graph

Select the file → **Generate Graph**. Watch chunking, embeddings, extraction, and load.

### 3.6 Explore results

- Preview the graph in the UI.
- In Neo4j Browser run: `MATCH (n) RETURN labels(n), count(*)`
- Try chat: *Which organizations are connected to major ports?*

### 3.7 Typical graph layers

| Layer | Purpose |
|-------|---------|
| Lexical graph | Document → Chunk (+ embeddings) |
| Entity graph | Domain entities and relationships |
| Links | Chunks connected to mentioned entities |

> Localhost Neo4j may not be reachable from the hosted app — use Section 2.2b only if needed.

## 2.2b Programmatic construction (notebook path)
The web app and this notebook share the same core idea: LangChain **`LLMGraphTransformer`**.
1. Wrap text as a LangChain `Document`
2. Run `LLMGraphTransformer` → `GraphDocument` objects
3. Write to Neo4j with `Neo4jGraph`
Lab data is tagged with **`CourseKG`** marker nodes for cleanup.
### Notebook steps (run in order)
| Step | What |
|------|------|
| **2.2b-1** | `Document` from corpus |
| **2.2b-2** | Configure open-schema transformer |
| **2.2b-3** | LLM extraction (slow on Ollama) |
| **2.2b-4** | Load into Neo4j |
| **2.2b-5** | Summary Cypher counts |


#### 2.2b-1 — One `Document` per article

After **Step 2a-helper**, each article is a separate `Document` (typically 300–2,500 characters). `LAB_MAX_ARTICLES` (default **5**) limits how many LLM calls Section 2.2b makes.



In [15]:
# 2.2b-1 — Build Document list (one article per Document)
documents = articles_to_documents(
    _all_articles,
    source_name=SAMPLE_TEXT_PATH.name,
    max_articles=LAB_MAX_ARTICLES,
)
print("Documents for open-schema extraction:", len(documents))
for d in documents[:3]:
    print(
        " ",
        d.metadata["title"],
        "| chars:",
        len(d.page_content),
    )



Documents for open-schema extraction: 5
  90_BC | chars: 364
  Heinrich_Hertz | chars: 504
  Olof_Palme | chars: 2663


#### 2.2b-2 — Configure `LLMGraphTransformer` (open schema)
With `allowed_nodes=None`, the LLM may invent labels freely (good for exploration, noisy for production). `ignore_tool_usage=True` avoids tool-calling paths that some local models do not support well.


In [16]:
# 2.2b-2 — Open-schema transformer
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=None,
    allowed_relationships=None,
    node_properties=False,
    ignore_tool_usage=True,
)
print("LLMGraphTransformer ready (open schema).")


LLMGraphTransformer ready (open schema).


/tmp/ipykernel_1113268/1194613304.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


#### 2.2b-3 — Run LLM extraction (per article)

> **Timing:** Each article is one Ollama call. With `LAB_MAX_ARTICLES=5`, expect several minutes on CPU.

> **Tip:** For maritime-themed triples, set `SAMPLE_TEXT_PATH` to `dbpedia_maritime_corpus.txt` in Step 0b.3.

If you see **0 nodes** for every article, verify `ollama serve`, model name, and `OLLAMA_MAX_TOKENS` (try `4096`).



In [17]:
# 2.2b-3 — Extract entities and relationships (per article)
graph_documents = []
for i, doc in enumerate(documents, start=1):
    title = doc.metadata.get("title", f"article_{i}")
    print(f"\n[{i}/{len(documents)}] Extracting: {title} ({len(doc.page_content)} chars)")
    batch = llm_transformer.convert_to_graph_documents([doc])
    graph_documents.extend(batch)
    gd_i = batch[0]
    print(f"  -> nodes: {len(gd_i.nodes)}, relationships: {len(gd_i.relationships)}")

total_nodes = sum(len(gd.nodes) for gd in graph_documents)
total_rels = sum(len(gd.relationships) for gd in graph_documents)
print(f"\nTotal across {len(graph_documents)} graph document(s): {total_nodes} nodes, {total_rels} relationships")

if total_nodes == 0:
    print(
        "WARNING: No nodes extracted. Check Ollama is running, increase OLLAMA_MAX_TOKENS, "
        "or reduce prompt size. Try SAMPLE_TEXT_PATH = dbpedia_maritime_corpus.txt"
    )
else:
    gd = graph_documents[0]
    print("\nSample from first non-empty batch (first 8 nodes/rels):")
    for n in gd.nodes[:8]:
        print("  Node:", n.id, n.type, n.properties)
    for r in gd.relationships[:8]:
        print("  Rel:", r.source.id, "->", r.type, "->", r.target.id)




[1/5] Extracting: 90_BC (364 chars)
  -> nodes: 3, relationships: 2

[2/5] Extracting: Heinrich_Hertz (504 chars)
  -> nodes: 5, relationships: 4

[3/5] Extracting: Olof_Palme (2663 chars)
  -> nodes: 5, relationships: 4

[4/5] Extracting: Seventh_Amendment_to_the_United_States_Constitution (1958 chars)
  -> nodes: 8, relationships: 7

[5/5] Extracting: Red-eye_effect (657 chars)
  -> nodes: 4, relationships: 4

Total across 5 graph document(s): 25 nodes, 21 relationships

Sample from first non-empty batch (first 8 nodes/rels):
  Node: The Year of the Consulship of Caesar and Lupus Event {}
  Node: Year 90 BC Time Period {}
  Node: 664 Ab urbe condita Date System {}
  Rel: Year 90 BC -> KNOWN_AS -> The Year of the Consulship of Caesar and Lupus
  Rel: Year 90 BC -> EQUIVALENT_TO -> 664 Ab urbe condita


#### 2.2b-4 — Connect to Neo4j and persist the graph
`Neo4jGraph.add_graph_documents()` creates nodes, relationships, and (with `include_source=True`) links back to the source document.


In [18]:
# 2.2b-4 — Write GraphDocuments to Neo4j
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
graph.query(
    '''
    MERGE (m:CourseKG {module: 'Module_8', lab: 'building_kg_with_llms'})
    SET m.updatedAt = datetime()
    RETURN m.module AS module
    '''
)
graph.add_graph_documents(graph_documents, include_source=True)
print("Graph documents written to Neo4j.")


Graph documents written to Neo4j.


#### 2.2b-5 — Summarize loaded structure
Quick counts by node label and relationship type — same checks you would run in Neo4j Browser after the web UI load.


In [19]:
# 2.2b-5 — Post-load summary queries
summary = graph.query(
    '''
    MATCH (n)
    WHERE NOT n:CourseKG
    RETURN labels(n)[0] AS label, count(*) AS count
    ORDER BY count DESC
    LIMIT 15
    '''
)
print("Node counts by label:")
for row in summary:
    print(f"  {row['label']}: {row['count']}")
rels = graph.query(
    '''
    MATCH ()-[r]->()
    RETURN type(r) AS relType, count(*) AS count
    ORDER BY count DESC
    LIMIT 15
    '''
)
print("\nRelationship counts:")
for row in rels:
    print(f"  {row['relType']}: {row['count']}")


Node counts by label:
  Document: 17
  Organization: 14
  Country: 10
  Port: 9
  Location: 6
  Person: 6
  LangChainLab: 5
  Canal: 5
  Concept: 5
  Chunk: 4
  Group: 2
  Action: 1
  Date System: 1
  Device: 1
  Right: 1

Relationship counts:
  MENTIONS: 56
  LOCATED_IN: 12
  OPERATES_IN: 10
  USES_ROUTE: 9
  OPERATED_BY: 7
  HAS_CHUNK: 4
  CONNECTS: 3
  AFFECTS: 2
  PROVED_THEORY_OF: 1
  LED_PARTY: 1
  EQUIVALENT_TO: 1
  MENTORED: 1
  MURDERED: 1
  PART_OF: 1
  WAS_PRIME_MINISTER_OF: 1


---

## 2.3 Customize the schema

Production graphs use a **controlled vocabulary** of labels and relationship types.

| Node labels | Examples |
|-------------|----------|
| `Port`, `Organization`, `Canal`, `Country`, `Location` | Port of Rotterdam, Maersk, Panama Canal |

| Relationship types | Pattern |
|--------------------|---------|
| `LOCATED_IN`, `OPERATED_BY`, `OPERATES_IN`, `USES_ROUTE`, `MANAGED_BY` | Domain-specific |

This mirrors **entity graph extraction settings** in the Neo4j LLM Graph Builder UI.

### Notebook steps

| Step | What |
|------|------|
| **2.3a** | Define allowed labels and relationship types |
| **2.3b** | Schema-guided extraction |
| **2.3c** | Load into Neo4j with `lab: schema_guided` |



#### 2.3a — Define the controlled schema
Restricting **`allowed_nodes`** and **`allowed_relationships`** reduces hallucinated labels and aligns with the Neo4j LLM Graph Builder schema UI.


In [20]:
# 2.3a — Schema vocabulary
SCHEMA_NODES = ["Port", "Organization", "Canal", "Country", "Location"]
SCHEMA_RELS = [
    "LOCATED_IN",
    "OPERATED_BY",
    "OPERATES_IN",
    "MANAGED_BY",
    "USES_ROUTE",
    "CONNECTS",
]
print("Allowed nodes:", SCHEMA_NODES)
print("Allowed relationships:", SCHEMA_RELS)


Allowed nodes: ['Port', 'Organization', 'Canal', 'Country', 'Location']
Allowed relationships: ['LOCATED_IN', 'OPERATED_BY', 'OPERATES_IN', 'MANAGED_BY', 'USES_ROUTE', 'CONNECTS']


#### 2.3b — Schema-guided extraction

Uses **`dbpedia_maritime_corpus.txt`** when available (ports, Maersk, canals, etc.). With Ollama, `strict_mode=False` so slightly mismatched labels are kept; OpenAI can use `strict_mode=True` and `node_properties`.

Compare totals with the open-schema run in Section 2.2b.



In [21]:
# 2.3b — Run schema-guided LLMGraphTransformer
# Prefer maritime articles for Port / Organization / Canal schema
_maritime_path = MODULE_DIR / "data" / "dbpedia_maritime_corpus.txt"
if _maritime_path.is_file():
    _schema_articles = parse_corpus_articles(_maritime_path.read_text(encoding="utf-8"))
    _schema_source = _maritime_path.name
else:
    _schema_articles = _all_articles
    _schema_source = SAMPLE_TEXT_PATH.name

schema_documents = articles_to_documents(
    _schema_articles,
    source_name=_schema_source,
    max_articles=LAB_SCHEMA_ARTICLES,
    course="Module_8_schema",
)
print(f"Schema extraction: {len(schema_documents)} article(s) from {_schema_source}")

_schema_kwargs = dict(
    llm=llm,
    allowed_nodes=SCHEMA_NODES,
    allowed_relationships=SCHEMA_RELS,
    ignore_tool_usage=True,
    # Ollama JSON mode: strict filtering often drops all triples if labels differ slightly
    strict_mode=(LLM_BACKEND == "openai"),
    additional_instructions=(
        "Focus on ports, shipping companies, canals, countries, and logistics locations. "
        "Use only the allowed node labels and relationship types from the schema."
    ),
)
if LLM_BACKEND == "openai":
    _schema_kwargs["node_properties"] = ["country", "date"]
else:
    print(
        "Note: node_properties omitted — Ollama runner does not support function calling. "
        "strict_mode=False so valid maritime triples are not all filtered out."
    )

schema_transformer = LLMGraphTransformer(**_schema_kwargs)
print("Running schema-guided extraction...")
schema_graph_documents = []
for i, doc in enumerate(schema_documents, start=1):
    title = doc.metadata.get("title", f"article_{i}")
    print(f"  [{i}/{len(schema_documents)}] {title}")
    batch = schema_transformer.convert_to_graph_documents([doc])
    schema_graph_documents.extend(batch)
    b = batch[0]
    print(f"      nodes={len(b.nodes)}, rels={len(b.relationships)}")

sgd_nodes = sum(len(gd.nodes) for gd in schema_graph_documents)
sgd_rels = sum(len(gd.relationships) for gd in schema_graph_documents)
print(f"Schema-guided totals: {sgd_nodes} nodes, {sgd_rels} relationships")

if sgd_nodes == 0:
    print("\nDebug: raw LLM output for first schema document (first 1500 chars):")
    _raw = schema_transformer.chain.invoke({"input": schema_documents[0].page_content})
    if not isinstance(_raw, str):
        _raw = getattr(_raw, "content", str(_raw))
    print(_raw[:1500])



Schema extraction: 8 article(s) from dbpedia_maritime_corpus.txt
Note: node_properties omitted — Ollama runner does not support function calling. strict_mode=False so valid maritime triples are not all filtered out.
Running schema-guided extraction...
  [1/8] Port_of_Singapore
      nodes=3, rels=2
  [2/8] Port_of_Rotterdam
      nodes=3, rels=2
  [3/8] Port_of_Busan
      nodes=3, rels=2
  [4/8] Maersk
      nodes=4, rels=2
  [5/8] Mediterranean_Shipping_Company
      nodes=3, rels=2
  [6/8] Panama_Canal
      nodes=3, rels=2
  [7/8] Suez_Canal
      nodes=5, rels=4
  [8/8] International_Maritime_Organization
      nodes=3, rels=2
Schema-guided totals: 27 nodes, 18 relationships


#### 2.3c — Load schema-guided graph
Metadata `lab: schema_guided` distinguishes this load from the open-schema lab for cleanup queries.


In [22]:
# 2.3c — Persist schema-guided GraphDocuments
graph.query(
    '''
    MERGE (m:CourseKG {module: 'Module_8', lab: 'schema_guided'})
    SET m.updatedAt = datetime()
    '''
)
for doc in schema_graph_documents:
    doc.source.metadata["lab"] = "schema_guided"
graph.add_graph_documents(schema_graph_documents, include_source=True)
print("Schema-guided graph loaded.")


Schema-guided graph loaded.


## 2.4 Complete the Graph

Your graph is **complete** when all of the following are true:

1. **Entities loaded** — nodes appear in Neo4j with expected labels
2. **Relationships loaded** — typed edges connect the right entities
3. **Provenance preserved** — source document/chunk links exist (when using `include_source=True`)
4. **Schema pass done** — schema-guided extraction finished without critical errors
5. **Sanity checks pass** — label counts and relationship counts look reasonable

### Completion checklist

- [ ] Run label count query (`MATCH (n) RETURN labels(n), count(*)`)
- [ ] Run relationship count query (`MATCH ()-[r]->() RETURN type(r), count(*)`)
- [ ] Spot-check 3 entities in Neo4j Browser graph view
- [ ] Run one domain question with Cypher in Section 03

If counts are zero or labels are inconsistent, revisit **2.3 Customize the schema** and re-run extraction.


---

# 03. Exploring your knowledge graph

## 3.1 Querying using Cypher

Use Cypher to answer questions **without** calling an LLM.

### Starter patterns

- `MATCH (n:Port) RETURN n LIMIT 10`
- `MATCH (a)-[r]->(b) RETURN a.id, type(r), b.id LIMIT 20`
- `MATCH p=(a)-[*1..3]-(b) RETURN p LIMIT 10`

#### 3.1a — Pattern: organization operates in port
Return explicit triples `(Organization)-[:OPERATES_IN]->(Port)`.


In [23]:
# 3.1a — Organization → Port
rows = graph.query(
    '''
    MATCH (o:Organization)-[r:OPERATES_IN]->(p:Port)
    RETURN o.id AS organization, p.id AS port
    LIMIT 20
    '''
)
if rows:
    for row in rows:
        print(row)
else:
    print("No OPERATES_IN rows yet — try open-schema extraction or reference seed graph.")


{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}
{'organization': 'Maersk', 'port': 'Port_of_Rotterdam'}


#### 3.1b — Pattern: port location and optional operator
Use **`OPTIONAL MATCH`** when a port may exist without a linked organization.


In [24]:
# 3.1b — Port, country, optional organization
for row in graph.query(
    '''
    MATCH (p:Port)-[:LOCATED_IN]->(c:Country)
    OPTIONAL MATCH (o:Organization)-[:OPERATES_IN]->(p)
    RETURN p.id AS port, c.id AS country, o.id AS organization
    LIMIT 15
    '''
):
    print(row)


{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Busan', 'country': 'South Korea', 'organization': None}
{'port': 'Port of Rotterdam', 'country': 'Netherlands', 'organization': None}
{'port': 'Port of Rotterdam', 'country': 'Netherlands', 'organization': None}
{'port': 'Port of Rotterdam', 'country': 'Netherlands', 'organization': None}
{'port': 'Port of Busan', 'country': 'South Korea', 'organization': None}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Rotterdam', 'country': 'Netherlands', 'organization': 'Maersk'}
{'port': 'Port_of_Singapore', 'country': 'Singapore', 'organization': None}
{'port': 'Port_of_Rotterdam', 'country': 'Netherla

#### 3.1c — Search by name fragment
Parameterized `CONTAINS` is useful for interactive exploration in notebooks.


In [25]:
# 3.1c — Find entities containing a term
SEARCH_TERM = "Rotterdam"
for row in graph.query(
    '''
    MATCH (n)
    WHERE n.id CONTAINS $term
    RETURN labels(n) AS labels, n.id AS id
    LIMIT 15
    ''',
    params={"term": SEARCH_TERM},
):
    print(row)


{'labels': ['Port'], 'id': 'Port_of_Rotterdam'}
{'labels': ['Port'], 'id': 'Port of Rotterdam'}
{'labels': ['Organization'], 'id': 'Port of Rotterdam Authority'}
{'labels': ['Port', 'LlamaIndexLab'], 'id': 'Port_of_Rotterdam'}
{'labels': ['Port', 'LangChainLab'], 'id': 'Port_of_Rotterdam'}


#### 3.1d — Shortest path between two entities
Uses variable-length paths `-[*..6]-`. If no path exists, Neo4j returns an empty result (not an error).


In [26]:
# 3.1d — Shortest path example
start_name = "Port_of_Rotterdam"
end_name = "Panama_Canal"
result = graph.query(
    '''
    MATCH (a {id: $start}), (b {id: $end})
    MATCH p = shortestPath((a)-[*..6]-(b))
    RETURN [n IN nodes(p) | coalesce(n.id, toString(id(n)))] AS path,
           [r IN relationships(p) | type(r)] AS relTypes
    ''',
    params={"start": start_name, "end": end_name},
)
if result:
    print("Path:", result[0]["path"])
    print("Relationship types:", result[0]["relTypes"])
else:
    print(f"No path found between {start_name} and {end_name}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=53, offset=139>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 139, 'line': 4, 'column': 53}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (a {id: $start}), (b {id: $end})\n    MATCH p = shortestPath((a)-[*..6]-(b))\n    RETURN [n IN nodes(p) | coalesce(n.id, toString(id(n)))] AS path,\n           [r IN relationships(p) | type(r)] AS relTypes\n    '


Path: ['Port_of_Rotterdam', 'Maersk', 'Panama_Canal']
Relationship types: ['OPERATES_IN', 'USES_ROUTE']


## 3.2 Explore the Knowledge Graph

Exploration means validating structure and discovering insights visually and programmatically.

### In Neo4j Browser / Bloom

1. Open your Neo4j instance (Aura, Desktop, or Docker Browser URL)
2. Run graph-style queries, for example:

```cypher
MATCH p=(:Port)-[*1..2]-()
RETURN p
LIMIT 25
```

3. Click nodes to inspect properties (`id`, `country`, etc.)

### What to look for

- Are port and organization entities connected as expected?
- Are relationship types consistent with your schema?
- Are there orphan nodes (no relationships)?

#### 3.2a — Orphan nodes (no relationships)
Orphans often indicate extraction noise or missing relationship types in your schema.


In [27]:
# 3.2a — Orphan node sample
orphan_nodes = graph.query(
    '''
    MATCH (n)
    WHERE NOT (n)--() AND NOT n:CourseKG
    RETURN labels(n) AS labels, coalesce(n.id, toString(id(n))) AS id
    LIMIT 20
    '''
)
print("Orphan nodes (sample):")
if orphan_nodes:
    for row in orphan_nodes:
        print(" ", row)
else:
    print("  None found in sample (good sign).")


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=57, offset=112>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 112, 'line': 4, 'column': 57}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (n)\n    WHERE NOT (n)--() AND NOT n:CourseKG\n    RETURN labels(n) AS labels, coalesce(n.id, toString(id(n))) AS id\n    LIMIT 20\n    '


Orphan nodes (sample):
  {'labels': ['Document'], 'id': 'bc8a4d5feaf6c9d0eb16543ecbc42866'}
  {'labels': ['Chunk'], 'id': 'f02c41ffe31773d071bc850d6582ba2e'}
  {'labels': ['Chunk'], 'id': 'c3e2bd57957511fc6b22b15be202adbb'}
  {'labels': ['LlamaIndexLab'], 'id': '74'}
  {'labels': ['Chunk'], 'id': 'f21032245792e8fafc7e94cbed7864f8'}
  {'labels': ['Chunk'], 'id': 'b2ed1b839b4fc452dc7170c595dc6864'}
  {'labels': ['LangChainLab'], 'id': '88'}


#### 3.2b — High-degree hub nodes
Nodes with many incident edges are often important entities (major ports, global organizations).


In [28]:
# 3.2b — Hub nodes by degree
hubs = graph.query(
    '''
    MATCH (n)-[r]-()
    WHERE NOT n:CourseKG
    RETURN labels(n)[0] AS label, coalesce(n.id, toString(id(n))) AS id, count(r) AS degree
    ORDER BY degree DESC
    LIMIT 10
    '''
)
print("Highest-degree nodes:")
for row in hubs:
    print(" ", row)


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=59, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 4, 'column': 59}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (n)-[r]-()\n    WHERE NOT n:CourseKG\n    RETURN labels(n)[0] AS label, coalesce(n.id, toString(id(n))) AS id, count(r) AS degree\n    ORDER BY degree DESC\n    LIMIT 10\n    '


Highest-degree nodes:
  {'label': 'Organization', 'id': 'Maersk', 'degree': 25}
  {'label': 'Port', 'id': 'Port_of_Rotterdam', 'degree': 13}
  {'label': 'Canal', 'id': 'Panama_Canal', 'degree': 10}
  {'label': 'Country', 'id': 'Netherlands', 'degree': 9}
  {'label': 'Document', 'id': '2db7a816bbf9605b322a8d33b95e3f16', 'degree': 8}
  {'label': 'Amendment', 'id': 'Seventh Amendment', 'degree': 7}
  {'label': 'Document', 'id': '2c97aaf101ec9a2a4e567093887cc0a8', 'degree': 6}
  {'label': 'Person', 'id': 'Olof Palme', 'degree': 5}
  {'label': 'Document', 'id': '9812009259f5f6f8806c4ab9584dc183', 'degree': 5}
  {'label': 'Person', 'id': 'Heinrich Rudolf Hertz', 'degree': 5}


### GraphRAG reminder (from 1.4 Use Cases)

| Capability | Benefit |
|------------|---------|
| Explicit relationships | Multi-hop reasoning with Cypher |
| Schema | Stable labels for agents and APIs |
| Provenance | Link answers to source chunks |
| GraphRAG | Chunks + graph context for LLM prompts |

Optional next step: **`neo4j-graphrag`** (`SimpleKGPipeline`) — see `NEO4J_SETUP.md` and `LLM_MODEL_SETUP.md`.

## 3.3 Create your Knowledge Graph

This is your capstone: build and validate a graph for your own domain.

### Project brief

1. Choose a short unstructured source (500–2000 words)
2. Define a schema (at least 4 node labels, 4 relationship types)
3. Build the graph (UI in **2.2** or code in **2.2b** + **2.3**)
4. Complete the graph using the checklist in **2.4**
5. Explore and query in **3.1** and **3.2**
6. Write 3 Cypher queries that answer real business questions

### Suggested domains

- Maritime logistics and ports
- Supply chain and manufacturing
- Healthcare providers and treatments
- Cybersecurity incidents and assets

### Deliverables

- Screenshot or export of graph visualization
- 3 Cypher queries + short interpretation of results
- 1 paragraph on schema choices and extraction quality

---

### Optional cleanup

In [29]:
CLEANUP = False

if CLEANUP:
    graph.query("MATCH (m:CourseKG {module: 'Module_8'}) DETACH DELETE m")
    print("Cleanup executed.")
else:
    print("Cleanup skipped. Set CLEANUP=True to remove CourseKG markers.")

Cleanup skipped. Set CLEANUP=True to remove CourseKG markers.


---

## Summary

You completed the full learning path:

- **01. Knowledge graphs** — concepts, benefits/challenges, exploration, use cases
- **02. LLM Graph Builder** — LLM construction flow, Neo4j UI, schema customization, graph completion
- **03. Exploring your knowledge graph** — Cypher querying, exploration, capstone project

**Further reading**

- [Creating Knowledge Graphs from Unstructured Data](https://neo4j.com/developer/genai-ecosystem/importing-graph-from-unstructured-data/)
- [Introduction to the Neo4j LLM Knowledge Graph Builder](https://neo4j.com/blog/developer/llm-knowledge-graph-builder/)

**Setup:** `NEO4J_SETUP.md` and `LLM_MODEL_SETUP.md`